In [1]:
# CELL 1: Install Dependencies
# ============================================================================
# IMPORTANT: After this cell finishes, the kernel will restart automatically.
# Once it restarts, run ALL remaining cells from Cell 2 onwards.
# Do NOT re-run Cell 1 after restart.
# ============================================================================
 
import os
import sys
 
# Check GPU availability
print("Checking GPU...")
os.system('nvidia-smi')
 
# Install required packages — let pip resolve versions freely to avoid conflicts
print("\nInstalling dependencies...")
print("="*80)
 
os.system("pip install -q transformers datasets accelerate scikit-learn")
 
print("✓ Dependencies installed")
print("\n*** KERNEL IS RESTARTING — wait for it, then run cells 2 onwards ***")
 
# Restart kernel to clear any binary incompatibility from pre-installed packages
import IPython
IPython.Application.instance().kernel.do_shutdown(True)

Checking GPU...
Wed Mar 18 21:04:41 2026       
+-----------------------------------------------------------------------------------------+
| NVIDIA-SMI 580.105.08             Driver Version: 580.105.08     CUDA Version: 13.0     |
+-----------------------------------------+------------------------+----------------------+
| GPU  Name                 Persistence-M | Bus-Id          Disp.A | Volatile Uncorr. ECC |
| Fan  Temp   Perf          Pwr:Usage/Cap |           Memory-Usage | GPU-Util  Compute M. |
|                                         |                        |               MIG M. |
|=========================================+========================+======================|
|   0  Tesla T4                       Off |   00000000:00:04.0 Off |                    0 |
| N/A   37C    P8              9W /   70W |       0MiB /  15360MiB |      0%      Default |
|                                         |                        |                  N/A |
+-------------------------------

{'status': 'ok', 'restart': True}

In [2]:
 #============================================================================
# CELL 2: Import Libraries
# ============================================================================
 
import pandas as pd
import numpy as np
import torch
from torch.utils.data import Dataset
from transformers import (
    AutoTokenizer,
    AutoModelForSequenceClassification,
    TrainingArguments,
    Trainer,
    EarlyStoppingCallback
)
from sklearn.metrics import accuracy_score, precision_recall_fscore_support, confusion_matrix
import warnings
warnings.filterwarnings('ignore')
 
print("PyTorch version:", torch.__version__)
print("CUDA available:", torch.cuda.is_available())
print("CUDA device:", torch.cuda.get_device_name(0) if torch.cuda.is_available() else "None")
 

PyTorch version: 2.9.0+cu126
CUDA available: True
CUDA device: Tesla T4


In [6]:
# ============================================================================
# CELL 3: Configuration
# ============================================================================
 
class Config:
    """Training configuration"""
    
    # Dataset paths (UPDATE THIS with your dataset path)
    DATASET_PATH = "/kaggle/input/datasets/nitishpatil2027/claim-detection-fever-dataset"  # ← CHANGE THIS
    
    # Model settings
    MODEL_NAME = "bert-base-uncased"  # You can also try: "roberta-base", "distilbert-base-uncased"
    MAX_LENGTH = 128
    
    # Training settings
    BATCH_SIZE = 32  # Increase to 64 if you have enough GPU memory
    LEARNING_RATE = 2e-5
    NUM_EPOCHS = 3
    WARMUP_STEPS = 500
    WEIGHT_DECAY = 0.01
    
    # Other settings
    OUTPUT_DIR = "./claim_detector_output"
    SEED = 42
 
config = Config()
 
print("\nConfiguration:")
print(f"  Model: {config.MODEL_NAME}")
print(f"  Batch size: {config.BATCH_SIZE}")
print(f"  Epochs: {config.NUM_EPOCHS}")
print(f"  Learning rate: {config.LEARNING_RATE}")
 


Configuration:
  Model: bert-base-uncased
  Batch size: 32
  Epochs: 3
  Learning rate: 2e-05


In [7]:
# ============================================================================
# CELL 4: Load Dataset
# ============================================================================
 
print("\nLoading dataset...")
print("="*80)
 
# Load CSVs
train_df = pd.read_csv(f"{config.DATASET_PATH}/claim_detection_train.csv")
val_df = pd.read_csv(f"{config.DATASET_PATH}/claim_detection_val.csv")
test_df = pd.read_csv(f"{config.DATASET_PATH}/claim_detection_test.csv")
 
print(f"Train: {len(train_df):,} examples")
print(f"Val:   {len(val_df):,} examples")
print(f"Test:  {len(test_df):,} examples")
 
# Check label distribution
print("\nLabel distribution (Train):")
print(train_df['label'].value_counts())
 
# Show samples
print("\nSample claims (label=1):")
print(train_df[train_df['label']==1]['text'].head(3).tolist())
 
print("\nSample non-claims (label=0):")
print(train_df[train_df['label']==0]['text'].head(3).tolist())
 
print("\n✓ Dataset loaded successfully")
 


Loading dataset...
Train: 280,713 examples
Val:   35,089 examples
Test:  35,090 examples

Label distribution (Train):
label
1    140357
0    140356
Name: count, dtype: int64

Sample claims (label=1):
['Bill Gates was born on November 3, 1912.', 'One Direction made Life.', 'Tylenol is only a brand of clothes.']

Sample non-claims (label=0):
['When did the meeting happen?', 'Where is the capital?', 'Examine climate']

✓ Dataset loaded successfully


In [8]:
# ============================================================================
# CELL 5: Create PyTorch Dataset
# ============================================================================
 
class ClaimDataset(Dataset):
    """Dataset for claim detection"""
    
    def __init__(self, texts, labels, tokenizer, max_length):
        self.texts = texts
        self.labels = labels
        self.tokenizer = tokenizer
        self.max_length = max_length
    
    def __len__(self):
        return len(self.texts)
    
    def __getitem__(self, idx):
        text = str(self.texts[idx])
        label = self.labels[idx]
        
        encoding = self.tokenizer(
            text,
            max_length=self.max_length,
            padding='max_length',
            truncation=True,
            return_tensors='pt'
        )
        
        return {
            'input_ids': encoding['input_ids'].flatten(),
            'attention_mask': encoding['attention_mask'].flatten(),
            'labels': torch.tensor(label, dtype=torch.long)
        }
 
print("✓ Dataset class defined")

✓ Dataset class defined


In [9]:
# ============================================================================
# CELL 6: Load Tokenizer and Create Datasets
# ============================================================================
 
print("\nLoading tokenizer...")
tokenizer = AutoTokenizer.from_pretrained(config.MODEL_NAME)
 
print("Creating datasets...")
train_dataset = ClaimDataset(
    train_df['text'].values,
    train_df['label'].values,
    tokenizer,
    config.MAX_LENGTH
)
 
val_dataset = ClaimDataset(
    val_df['text'].values,
    val_df['label'].values,
    tokenizer,
    config.MAX_LENGTH
)
 
test_dataset = ClaimDataset(
    test_df['text'].values,
    test_df['label'].values,
    tokenizer,
    config.MAX_LENGTH
)
 
print(f"✓ Datasets created:")
print(f"  Train: {len(train_dataset):,}")
print(f"  Val:   {len(val_dataset):,}")
print(f"  Test:  {len(test_dataset):,}")


Loading tokenizer...


config.json:   0%|          | 0.00/570 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/48.0 [00:00<?, ?B/s]

vocab.txt: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

Creating datasets...
✓ Datasets created:
  Train: 280,713
  Val:   35,089
  Test:  35,090


In [16]:
# ============================================================================
# CELL 7: Load Model
# ============================================================================
 
print("\nLoading model...")
model = AutoModelForSequenceClassification.from_pretrained(
    config.MODEL_NAME,
    num_labels=2
)
 
# Move to GPU
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
model.to(device)
 
print(f"✓ Model loaded on {device}")
print(f"  Parameters: {sum(p.numel() for p in model.parameters()):,}")

# Verify LayerNorm weights are not random (should not be all 1s/0s)
layer = model.bert.encoder.layer[0].attention.output.LayerNorm
print("LayerNorm weight sample:", layer.weight[:5].tolist())
print("LayerNorm bias sample:", layer.bias[:5].tolist())

# If weights are pretrained you'll see varied values like:
# [0.932, 1.043, 0.876, 1.102, 0.954]  ← good, pretrained values
# [1.0, 1.0, 1.0, 1.0, 1.0]            ← bad, randomly initialized
 


Loading model...


Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

BertForSequenceClassification LOAD REPORT from: bert-base-uncased
Key                                        | Status     | 
-------------------------------------------+------------+-
cls.seq_relationship.bias                  | UNEXPECTED | 
cls.predictions.transform.LayerNorm.bias   | UNEXPECTED | 
cls.predictions.transform.LayerNorm.weight | UNEXPECTED | 
cls.predictions.transform.dense.bias       | UNEXPECTED | 
cls.predictions.transform.dense.weight     | UNEXPECTED | 
cls.seq_relationship.weight                | UNEXPECTED | 
cls.predictions.bias                       | UNEXPECTED | 
classifier.weight                          | MISSING    | 
classifier.bias                            | MISSING    | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING	:those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.


✓ Model loaded on cuda
  Parameters: 109,483,778
LayerNorm weight sample: [0.9803407788276672, 0.9599689841270447, 0.9636898636817932, 0.9603652954101562, 0.9801324009895325]
LayerNorm bias sample: [0.25779154896736145, -0.030778534710407257, -0.2772696912288666, -0.38847702741622925, 0.3684176504611969]


In [17]:
# Check what the final metrics looked like
print(trainer.state.best_metric)
print(trainer.state.log_history[-10:])  # last 10 log entries

1.0
[{'loss': 0.00031026555225253103, 'grad_norm': 0.0018463818123564124, 'learning_rate': 1.8579891003870155e-05, 'epoch': 0.31912468657396853, 'step': 1400}, {'loss': 5.660393740981817e-05, 'grad_norm': 0.001185513217933476, 'learning_rate': 1.8421925598293976e-05, 'epoch': 0.34191930704353773, 'step': 1500}, {'eval_loss': 0.0011356659233570099, 'eval_accuracy': 0.9999430020804241, 'eval_precision': 1.0, 'eval_recall': 0.9998860009119928, 'eval_f1': 0.9999429972068631, 'eval_runtime': 153.7567, 'eval_samples_per_second': 228.211, 'eval_steps_per_second': 1.789, 'epoch': 0.34191930704353773, 'step': 1500}, {'loss': 4.686173517256975e-05, 'grad_norm': 0.0012108454247936606, 'learning_rate': 1.8263960192717797e-05, 'epoch': 0.3647139275131069, 'step': 1600}, {'loss': 3.775762626901269e-05, 'grad_norm': 0.0009529778035357594, 'learning_rate': 1.8105994787141615e-05, 'epoch': 0.3875085479826761, 'step': 1700}, {'loss': 3.553394228219986e-05, 'grad_norm': 0.0009013566887006164, 'learning_r

In [11]:
# ============================================================================
# CELL 8: Define Metrics
# ============================================================================
 
def compute_metrics(eval_pred):
    """Compute evaluation metrics"""
    predictions, labels = eval_pred
    predictions = np.argmax(predictions, axis=1)
    
    precision, recall, f1, _ = precision_recall_fscore_support(
        labels, predictions, average='binary'
    )
    acc = accuracy_score(labels, predictions)
    
    return {
        'accuracy': acc,
        'precision': precision,
        'recall': recall,
        'f1': f1
    }
 
print("✓ Metrics function defined")
 

✓ Metrics function defined


In [12]:
# ============================================================================
# CELL 9: Training Arguments
# ============================================================================
 
# Detect which parameter name this transformers version supports
import transformers
transformers_version = tuple(int(x) for x in transformers.__version__.split(".")[:2])
eval_strategy_key = "evaluation_strategy" if transformers_version < (4, 41) else "eval_strategy"
 
training_args = TrainingArguments(
    output_dir=config.OUTPUT_DIR,
    
    # Training hyperparameters
    num_train_epochs=config.NUM_EPOCHS,
    per_device_train_batch_size=config.BATCH_SIZE,
    per_device_eval_batch_size=config.BATCH_SIZE * 2,
    learning_rate=config.LEARNING_RATE,
    weight_decay=config.WEIGHT_DECAY,
    warmup_steps=config.WARMUP_STEPS,
    
    # Logging
    logging_dir=f"{config.OUTPUT_DIR}/logs",
    logging_steps=100,
    
    # Evaluation — works on both old and new transformers
    **{eval_strategy_key: "steps"},
    eval_steps=500,
    save_strategy="steps",
    save_steps=500,
    save_total_limit=2,
    load_best_model_at_end=True,
    metric_for_best_model="f1",
    
    # Other
    fp16=True,  # Mixed precision training
    dataloader_num_workers=2,
    report_to="none",  # Disable wandb
    seed=config.SEED,
)
 
print("✓ Training arguments configured")

`logging_dir` is deprecated and will be removed in v5.2. Please set `TENSORBOARD_LOGGING_DIR` instead.


✓ Training arguments configured


In [13]:
# ============================================================================
# CELL 10: Initialize Trainer
# ============================================================================
 
trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=train_dataset,
    eval_dataset=val_dataset,
    compute_metrics=compute_metrics,
    callbacks=[EarlyStoppingCallback(early_stopping_patience=3)]
)
 
print("✓ Trainer initialized")

✓ Trainer initialized


In [14]:
# ============================================================================
# CELL 11: Train Model (THIS TAKES 2-3 HOURS)
# ============================================================================
 
print("\n" + "="*80)
print("STARTING TRAINING")
print("="*80)
print("\nThis will take approximately 2-3 hours on T4 GPU")
print("You can monitor progress below...")
print("="*80 + "\n")
 
# Train
trainer.train()
 
print("\n" + "="*80)
print("✓ TRAINING COMPLETE!")
print("="*80)


STARTING TRAINING

This will take approximately 2-3 hours on T4 GPU
You can monitor progress below...



Step,Training Loss,Validation Loss,Accuracy,Precision,Recall,F1
500,0.002093,0.000244,1.000000,1.000000,1.000000,1.000000
1000,0.000098,0.000055,1.000000,1.000000,1.000000,1.000000
1500,0.000057,0.001136,0.999943,1.000000,0.999886,0.999943
2000,0.000027,0.000370,0.999972,1.000000,0.999943,0.999971


Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

There were missing keys in the checkpoint model loaded: ['bert.embeddings.LayerNorm.weight', 'bert.embeddings.LayerNorm.bias', 'bert.encoder.layer.0.attention.output.LayerNorm.weight', 'bert.encoder.layer.0.attention.output.LayerNorm.bias', 'bert.encoder.layer.0.output.LayerNorm.weight', 'bert.encoder.layer.0.output.LayerNorm.bias', 'bert.encoder.layer.1.attention.output.LayerNorm.weight', 'bert.encoder.layer.1.attention.output.LayerNorm.bias', 'bert.encoder.layer.1.output.LayerNorm.weight', 'bert.encoder.layer.1.output.LayerNorm.bias', 'bert.encoder.layer.2.attention.output.LayerNorm.weight', 'bert.encoder.layer.2.attention.output.LayerNorm.bias', 'bert.encoder.layer.2.output.LayerNorm.weight', 'bert.encoder.layer.2.output.LayerNorm.bias', 'bert.encoder.layer.3.attention.output.LayerNorm.weight', 'bert.encoder.layer.3.attention.output.LayerNorm.bias', 'bert.encoder.layer.3.output.LayerNorm.weight', 'bert.encoder.layer.3.output.LayerNorm.bias', 'bert.encoder.layer.4.attention.output.La


✓ TRAINING COMPLETE!


In [18]:
# ============================================================================
# CELL 12: Evaluate on Test Set
# ============================================================================
 
print("\nEvaluating on test set...")
print("="*80)
 
test_results = trainer.evaluate(test_dataset)
 
print("\nTEST SET RESULTS:")
print("="*80)
print(f"Accuracy:  {test_results['eval_accuracy']:.4f}")
print(f"Precision: {test_results['eval_precision']:.4f}")
print(f"Recall:    {test_results['eval_recall']:.4f}")
print(f"F1 Score:  {test_results['eval_f1']:.4f}")
print("="*80)
 
# Confusion Matrix
predictions = trainer.predict(test_dataset)
pred_labels = np.argmax(predictions.predictions, axis=1)
true_labels = test_df['label'].values
 
cm = confusion_matrix(true_labels, pred_labels)
 
print("\nConfusion Matrix:")
print("                Predicted")
print("              Not-Claim  Claim")
print(f"Actual Not-Claim  {cm[0][0]:6d}  {cm[0][1]:6d}")
print(f"       Claim      {cm[1][0]:6d}  {cm[1][1]:6d}")
 


Evaluating on test set...



TEST SET RESULTS:
Accuracy:  1.0000
Precision: 1.0000
Recall:    1.0000
F1 Score:  1.0000

Confusion Matrix:
                Predicted
              Not-Claim  Claim
Actual Not-Claim   17545       0
       Claim           0   17545


In [24]:
# ============================================================================
# CELL 13: Save Model
# ============================================================================
import os
import json
print("\nSaving model...")
 
# Create output directory
output_path = "./claim_detector_final"
os.makedirs(output_path, exist_ok=True)
 
# Save model and tokenizer
trainer.save_model(output_path)
tokenizer.save_pretrained(output_path)
 
# Save training results
import json
results = {
    'test_accuracy': float(test_results['eval_accuracy']),
    'test_precision': float(test_results['eval_precision']),
    'test_recall': float(test_results['eval_recall']),
    'test_f1': float(test_results['eval_f1']),
    'model_name': config.MODEL_NAME,
    'epochs': config.NUM_EPOCHS,
}
 
with open(f"{output_path}/results.json", 'w') as f:
    json.dump(results, indent=2, fp=f)
 
# Create README
readme = f"""# Claim Detection Model
 
Trained on FEVER dataset for binary claim detection.
 
## Model Details
- Base model: {config.MODEL_NAME}
- Epochs: {config.NUM_EPOCHS}
- Training examples: {len(train_df):,}
 
## Performance
- Accuracy: {test_results['eval_accuracy']:.4f}
- Precision: {test_results['eval_precision']:.4f}
- Recall: {test_results['eval_recall']:.4f}
- F1 Score: {test_results['eval_f1']:.4f}
 
## Usage
```python
from transformers import pipeline
 
classifier = pipeline("text-classification", model="./claim_detector_final")
result = classifier("India's GDP grew 8% in 2024")
print(result)
# [{{'label': 'LABEL_1', 'score': 0.95}}]  # LABEL_1 = claim
```
 
## Labels
- LABEL_0: Not a claim
- LABEL_1: Is a claim
"""
 
with open(f"{output_path}/README.md", 'w') as f:
    f.write(readme)
 
print(f"\n✓ Model saved to: {output_path}")
print("\nSaved files:")
print("  - config.json")
print("  - pytorch_model.bin (or model.safetensors)")
print("  - tokenizer files")
print("  - results.json")
print("  - README.md")
 


Saving model...


Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]


✓ Model saved to: ./claim_detector_final

Saved files:
  - config.json
  - pytorch_model.bin (or model.safetensors)
  - tokenizer files
  - results.json
  - README.md


In [25]:
# ============================================================================
# CELL 14: Test Saved Model
# ============================================================================
 
print("\nTesting saved model...")
 
from transformers import pipeline
 
classifier = pipeline("text-classification", model=output_path)
 
test_cases = [
    "India's GDP grew 8% in 2024",
    "What is the GDP growth rate?",
    "I think the economy is good",
    "The unemployment rate fell to 5%"
]
 
print("\nTest predictions:")
print("="*80)
 
for text in test_cases:
    result = classifier(text)[0]
    is_claim = result['label'] == 'LABEL_1'
    
    print(f"\nText: {text}")
    print(f"  Prediction: {'CLAIM' if is_claim else 'NOT CLAIM'}")
    print(f"  Confidence: {result['score']:.3f}")
 
print("\n" + "="*80)
print("✓ Model is working correctly!")
 


Testing saved model...


Loading weights:   0%|          | 0/201 [00:00<?, ?it/s]


Test predictions:

Text: India's GDP grew 8% in 2024
  Prediction: CLAIM
  Confidence: 1.000

Text: What is the GDP growth rate?
  Prediction: NOT CLAIM
  Confidence: 1.000

Text: I think the economy is good
  Prediction: NOT CLAIM
  Confidence: 0.992

Text: The unemployment rate fell to 5%
  Prediction: CLAIM
  Confidence: 0.992

✓ Model is working correctly!


In [26]:
from transformers import pipeline
classifier = pipeline("text-classification", model="./claim_detector_final")

custom_tests = [
    # These should be CLAIM
    "India launched a mission to Mars in 2023",
    "The population of China exceeds 1.4 billion",
    "Tesla was founded in 2003",
    
    # These should be NOT CLAIM  
    "Do you know when Tesla was founded?",
    "I feel like electric cars are probably better",
    "What do you think about space missions?",
    
    # Edge cases — harder
    "Some people believe the earth is flat",      # hedged claim
    "Scientists say coffee might cause cancer",   # reported claim
]

for text in custom_tests:
    result = classifier(text)[0]
    label = "CLAIM" if result['label'] == 'LABEL_1' else "NOT CLAIM"
    print(f"{label} ({result['score']:.3f}) | {text}")

Loading weights:   0%|          | 0/201 [00:00<?, ?it/s]

CLAIM (1.000) | India launched a mission to Mars in 2023
CLAIM (1.000) | The population of China exceeds 1.4 billion
CLAIM (1.000) | Tesla was founded in 2003
NOT CLAIM (1.000) | Do you know when Tesla was founded?
CLAIM (1.000) | I feel like electric cars are probably better
NOT CLAIM (1.000) | What do you think about space missions?
CLAIM (0.999) | Some people believe the earth is flat
CLAIM (0.996) | Scientists say coffee might cause cancer


In [27]:
# ============================================================================
# CELL 15: Create Download Package
# ============================================================================
 
print("\nCreating download package...")
 
os.system("zip -r claim_detector_final.zip claim_detector_final/")
 
print("\n✓ Package created: claim_detector_final.zip")
print("\nTo download:")
print("  1. Click on 'claim_detector_final.zip' in the output files")
print("  2. Click the download button")
print("  3. Extract on your local machine to:")
print("     fact-verification-system/models/claim_detector/final/")


Creating download package...
  adding: claim_detector_final/ (stored 0%)
  adding: claim_detector_final/model.safetensors (deflated 7%)
  adding: claim_detector_final/README.md (deflated 36%)
  adding: claim_detector_final/results.json (deflated 35%)
  adding: claim_detector_final/tokenizer_config.json (deflated 42%)
  adding: claim_detector_final/training_args.bin (deflated 53%)
  adding: claim_detector_final/config.json (deflated 53%)
  adding: claim_detector_final/tokenizer.json (deflated 71%)

✓ Package created: claim_detector_final.zip

To download:
  1. Click on 'claim_detector_final.zip' in the output files
  2. Click the download button
  3. Extract on your local machine to:
     fact-verification-system/models/claim_detector/final/


In [28]:
# ============================================================================
# CELL 16: Summary
# ============================================================================
 
print("\n" + "="*80)
print("TRAINING COMPLETE - SUMMARY")
print("="*80)
 
print(f"\nModel: {config.MODEL_NAME}")
print(f"Training time: ~2-3 hours")
print(f"Training examples: {len(train_df):,}")
 
print(f"\nFinal Test Results:")
print(f"  Accuracy:  {test_results['eval_accuracy']:.4f}")
print(f"  Precision: {test_results['eval_precision']:.4f}")
print(f"  Recall:    {test_results['eval_recall']:.4f}")
print(f"  F1 Score:  {test_results['eval_f1']:.4f}")
 
print(f"\nModel saved to: {output_path}")
print(f"Download file: claim_detector_final.zip")
 
print("\n" + "="*80)
print("NEXT STEPS:")
print("="*80)
print("1. Download claim_detector_final.zip")
print("2. Extract to: models/claim_detector/final/")
print("3. Update configs/nlp_config.yaml:")
print("     models:")
print("       claim_detector:")
print("         use_trained: true")
print("4. Test with: python tests/test_trained_claim_detector.py")
print("="*80)


TRAINING COMPLETE - SUMMARY

Model: bert-base-uncased
Training time: ~2-3 hours
Training examples: 280,713

Final Test Results:
  Accuracy:  1.0000
  Precision: 1.0000
  Recall:    1.0000
  F1 Score:  1.0000

Model saved to: ./claim_detector_final
Download file: claim_detector_final.zip

NEXT STEPS:
1. Download claim_detector_final.zip
2. Extract to: models/claim_detector/final/
3. Update configs/nlp_config.yaml:
     models:
       claim_detector:
         use_trained: true
4. Test with: python tests/test_trained_claim_detector.py
